In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from pyspark.sql import functions as F

df_gold = spark.table("dbw_routemind_euskadi_dev.gold.weather_features")

In [0]:
# filter for tagging historical data in target column outdoor/indoor
df_gold = df_gold.withColumn(
    "is_outdoor",
    F.when(
        (F.col("temp_avg_mun") >= 10.0) &  # good temperature for outdoor activities
        (F.col("temp_max_mun") < 40.0) &
        (F.col("temp_min_mun") > 0.0) &
        (F.col("precip_avg_mun") < 2.0) &  # No significant rainfall
        (F.col("PM2_5_avg_mun") < 30.0) &  # Good air quality
        (F.col("PM_10_avg_mun") < 50.0), 1 
    ).otherwise(0)
)

In [0]:
display(df_gold.groupBy("is_outdoor").count())

In [0]:
# pandas to apply transformations in training dataset
pdf = df_gold.toPandas()

# Cyclic Time Transformation for model to identify the start and end of a year
pdf['date'] = pd.to_datetime(pdf['date'])
pdf['day_of_year'] = pdf['date'].dt.dayofyear
pdf['day_sin'] = np.sin(2 * np.pi * pdf['day_of_year'] / 365.0)
pdf['day_cos'] = np.cos(2 * np.pi * pdf['day_of_year'] / 365.0)

# 2. Unique geographical key and geographic transformation
pdf['unique_municipality'] = pdf['countyId'].astype(str) + "_" + pdf['municipalityCode'].astype(str)
pdf = pd.get_dummies(pdf, columns=['unique_municipality'], drop_first=True)


In [0]:
#definition of predicted and target variables
features = [col for col in pdf.columns if col.startswith('unique_municipality_')] + ['day_sin', 'day_cos']

X = pdf[features]
y = pdf['is_outdoor']

In [0]:
print(features)

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Experiment in MLflow
with mlflow.start_run(run_name="Weather_Outdoor_Classifier"):
    
    # Model definition and training
    clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    clf.fit(X_train, y_train)
    
    # Predictions
    y_pred = clf.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    
    # Logging parameters and metrics in MLflow
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score_outdoor", report['1']['f1-score'])
    
    # logging model artefact
    mlflow.sklearn.log_model(clf, "weather_profile_model")
    
    print(f"Model trained using Accuracy: {acc:.4f}")

In [0]:
import mlflow.sklearn
import pandas as pd
import numpy as np

# get URI model. 
run_id = "157824302138461ea930bde970143be1"
model_uri = f"runs:/{run_id}/weather_profile_model"

print(f"loading model from: {model_uri}")
loaded_model = mlflow.sklearn.load_model(model_uri)

# Column consistency validation
# get the features on which the model was trained
training_cols = loaded_model.feature_names_in_
print(f"The model expects {len(training_cols)} predicted variables.")

# test with mock data
test_day = 299 # day 299 of the year
df_test = pd.DataFrame(0, index=[0], columns=training_cols)

# assign the transformed time value
df_test['day_sin'] = np.sin(2 * np.pi * test_day / 365.0)
df_test['day_cos'] = np.cos(2 * np.pi * test_day / 365.0)

# Unique geographical key
cod_municipality_test = 'unique_municipality_48_001'
if cod_municipality_test in df_test.columns:
    df_test[cod_municipality_test] = 1

# 4. Generación de la probabilidad
probability = loaded_model.predict_proba(df_test)
print(f"Validation result:")
print(f"probability Indoor (0): {probability[0][0]:.4f}")
print(f"probability Outdoor (1): {probability[0][1]:.4f}")